1. Load Data

In [ ]:
import pandas as pd

COLUMNS = ["city", "price", "location", "beds", "baths", "size", "title"]

combined = pd.read_csv("../data/all_listings_raw.csv", names=COLUMNS)

print("Shape:", combined.shape)
print("\nFirst 5 rows:")
combined.head()

2. Parse Price

In [ ]:
def parse_price(price_str):
    if pd.isnull(price_str):
        return None
    price_str = str(price_str).strip()
    
    # extract numeric part and unit
    if 'Arab' in price_str:
        return float(price_str.replace('Arab', '').strip()) * 1_000_000_000
    elif 'Crore' in price_str:
        return float(price_str.replace('Crore', '').strip()) * 10_000_000
    elif 'Lakh' in price_str:
        return float(price_str.replace('Lakh', '').strip()) * 100_000
    elif 'Thousand' in price_str:
        return float(price_str.replace('Thousand', '').strip()) * 1_000
    else:
        return None

combined['price_pkr'] = combined['price'].apply(parse_price)

# verify all formats
print("Price parsing verification:")
combined[['price', 'price_pkr']].drop_duplicates().dropna().sort_values('price_pkr', ascending=False).head(20)

3. Parse Size

In [ ]:
# conversion factors
# 1 Kanal     = 20 Marla
# 1 Sq. Yd.   = 1/25.2929 Marla
# 1 Sq. Ft.   = 1/272.25 Marla

def parse_size(size_str):
    if pd.isnull(size_str):
        return None
    size_str = str(size_str).strip().replace(',', '')  # fix 1,000 -> 1000
    
    try:
        if 'Kanal' in size_str:
            return round(float(size_str.replace('Kanal', '').strip()) * 20, 2)
        elif 'Marla' in size_str:
            return round(float(size_str.replace('Marla', '').strip()), 2)
        elif 'Sq. Yd.' in size_str:
            return round(float(size_str.replace('Sq. Yd.', '').strip()) / 25.2929, 2)
        elif 'Sq. Ft.' in size_str:
            return round(float(size_str.replace('Sq. Ft.', '').strip()) / 272.25, 2)
        else:
            return None
    except ValueError:
        return None

combined['size_marla'] = combined['size'].apply(parse_size)


# verify all formats
print("Size parsing verification:")
combined[['size', 'size_marla']].drop_duplicates().dropna().sort_values('size_marla', ascending=False).head(10)

4. Parse Location

In [ ]:
def split_location(location, part):
    if pd.isnull(location):
        return None
    parts = location.split(',')
    if part == 'neighbourhood':
        return parts[0].strip()
    elif part == 'area':
        return parts[-1].strip() if len(parts) > 1 else None

combined['neighbourhood'] = combined['location'].apply(lambda x: split_location(x, 'neighbourhood'))
combined['area']          = combined['location'].apply(lambda x: split_location(x, 'area'))

print("Location split verification:")
combined[['location', 'neighbourhood', 'area']].head(10)

5. Derived Metrics

In [ ]:
combined['price_per_marla'] = (combined['price_pkr'] / combined['size_marla']).round(2)

print("Price per Marla sample:")
combined[['city', 'neighbourhood', 'price', 'size_marla', 'price_per_marla']].sort_values('price_per_marla', ascending=False).head(30)

6. Handle Missing Values

In [ ]:
# convert to numeric first in case they were read as strings
combined['beds']  = pd.to_numeric(combined['beds'],  errors='coerce')
combined['baths'] = pd.to_numeric(combined['baths'], errors='coerce')

# fill by size group first
combined['beds']  = combined.groupby('size_marla')['beds'].transform(
    lambda x: x.fillna(x.median())
)
combined['baths'] = combined.groupby('size_marla')['baths'].transform(
    lambda x: x.fillna(x.median())
)

# fallback to overall median
combined['beds']  = combined['beds'].fillna(combined['beds'].median())
combined['baths'] = combined['baths'].fillna(combined['baths'].median())

print("Missing after fix:")
print(combined[['beds', 'baths']].isnull().sum())

7. Validate

In [ ]:
print("=== Shape ===")
print(combined.shape)

print("\n=== Data Types ===")
print(combined.dtypes)

print("\n=== Missing Values ===")
print(combined.isnull().sum())

print("\n=== Stats ===")
combined[['price_pkr', 'size_marla', 'price_per_marla', 'beds', 'baths']].describe()

8. Save

In [ ]:
# combined clean file
combined.to_csv("../data/all_listings_clean.csv", index=False)

# individual city files
for city in ['islamabad', 'lahore', 'karachi']:
    city_df = combined[combined['city'] == city]
    city_df.to_csv(f"../data/{city}_listings_clean.csv", index=False)
    print(f"{city}: {len(city_df)} listings saved")

print("\nAll files saved successfully.")